In [1]:
!pip install jiwer librosa evaluate noisereduce

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.0.13 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cudnn-cu12==9.1.0.70; 

In [3]:
!pip install -U datasets

In [4]:
#!pip install -q transformers==4.35.2
#!pip install -q datasets==2.15.0
#!pip install -q jiwer==3.0.3
#!pip install -q evaluate==0.4.1

In [5]:
#######################################################
###                     Import & Configuration      ###
#######################################################
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
import soundfile as sf
import base64
import io
import tempfile
import noisereduce as nr
import librosa
import soundfile as sf
import os
import IPython.display as ipd
from pathlib import Path
import evaluate
import warnings
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm import tqdm
from IPython.display import display, Audio, HTML
from datasets import load_dataset, Dataset, Audio
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback
)
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union
import evaluate
import jiwer
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.preprocessing import label_binarize

# Désactivation des warnings
warnings.filterwarnings('ignore')

# Configuration de base
MODEL_NAME = "facebook/wav2vec2-xls-r-300m"
DATASET_NAME = "IndabaxSenegal/asr-wolof-dataset"
DATASET_TEST_NAME = "IndabaxSenegal/asr-wolof-dataset-test"
OUTPUT_DIR = "./wav2vec2-wolof-results"
MAX_INPUT_LENGTH = 18  # secondes
MIN_INPUT_LENGTH = 1 # secondes
MAX_TOKENS = 310
BATCH_SIZE_TRAIN = 8  # Réduit de 16 à 8 pour aligner avec le premier code
BATCH_SIZE_EVAL = 8
NUM_EPOCHS = 16
LEARNING_RATE = 1e-4
WARMUP_STEPS = 1000
WEIGHT_DECAY = 0.005
SEED = 42

# Early Stopping Configuration
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.01

2025-06-23 12:36:19.160304: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750682179.368832      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750682179.439590      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [6]:
#######################################################
###                     Preparation Dataset         ###
#######################################################

# Configuration du seed pour la reproductibilité
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# Chargement des données
print("Chargement des données...")
dataset = load_dataset(DATASET_NAME, split="train")
dataset_test = load_dataset(DATASET_TEST_NAME, split="test")

# Split des données (90% train, 10% validation)
dataset = dataset.train_test_split(test_size=0.1, seed=SEED)

# Nettoyage des colonnes inutiles
dataset = dataset.remove_columns(["duration", "file_name", "path"])
dataset_test = dataset_test.remove_columns(["duration", "file_name", "path"])

# Normalisation du taux d'échantillonnage
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
dataset_test = dataset_test.cast_column("audio", Audio(sampling_rate=16_000))

Chargement des données...


README.md:   0%|          | 0.00/438 [00:00<?, ?B/s]

train-00000-of-00014.parquet:   0%|          | 0.00/503M [00:00<?, ?B/s]

train-00001-of-00014.parquet:   0%|          | 0.00/383M [00:00<?, ?B/s]

train-00002-of-00014.parquet:   0%|          | 0.00/535M [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/405 [00:00<?, ?B/s]

test-00000-of-00003.parquet:   0%|          | 0.00/301M [00:00<?, ?B/s]

test-00001-of-00003.parquet:   0%|          | 0.00/384M [00:00<?, ?B/s]

test-00002-of-00003.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
#######################################################
###                     Text Preprocessing         ###
#######################################################

# Nettoyage du texte - exactement comme dans le premier code
chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\"\%\'\"\�]'

def remove_special_characters(batch):
    batch["text"] = re.sub(chars_to_ignore_regex, '', batch["text"]).lower() + " "
    return batch

dataset = dataset.map(remove_special_characters)

Map:   0%|          | 0/11700 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

In [8]:
#######################################################
###          Tokenizer & Feature Extraction         ###
#######################################################
# 1. D'abord, créer le vocabulaire original (si pas déjà fait)
def create_original_vocab(dataset_train, dataset_test):
    """Crée le vocabulaire original comme référence"""
    def extract_all_chars(batch):
        all_text = " ".join(batch["text"])
        vocab = list(set(all_text))
        return {"vocab": [vocab], "all_text": [all_text]}

    vocab_train = dataset_train.map(
        extract_all_chars,
        batched=True,
        batch_size=-1,
        keep_in_memory=True,
        remove_columns=dataset_train.column_names
    )

    vocab_test = dataset_test.map(
        extract_all_chars,
        batched=True,
        batch_size=-1,
        keep_in_memory=True,
        remove_columns=dataset_test.column_names
    )

    all_chars = set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0])

    # Création du vocabulaire original (sans filtrage)
    original_vocab_dict = {char: idx for idx, char in enumerate(sorted(all_chars))}

    # Post-traitement standard
    if ' ' in original_vocab_dict:
        original_vocab_dict['|'] = original_vocab_dict[' ']
        del original_vocab_dict[' ']

    original_vocab_dict['[UNK]'] = len(original_vocab_dict)
    original_vocab_dict['[PAD]'] = len(original_vocab_dict)

    return original_vocab_dict

# Fonction pour nettoyer et optimiser le vocabulaire
def clean_and_optimize_vocab(dataset_train, dataset_test):
    """
    Crée un vocabulaire optimisé pour l'ASR en wolof
    """

    # 1. Extraction de tous les caractères des datasets
    def extract_all_chars(batch):
        all_text = " ".join(batch["text"])
        vocab = list(set(all_text))
        return {"vocab": [vocab], "all_text": [all_text]}

    vocab_train = dataset_train.map(
        extract_all_chars,
        batched=True,
        batch_size=-1,
        keep_in_memory=True,
        remove_columns=dataset_train.column_names
    )

    vocab_test = dataset_test.map(
        extract_all_chars,
        batched=True,
        batch_size=-1,
        keep_in_memory=True,
        remove_columns=dataset_test.column_names
    )

    # 2. Combinaison des vocabulaires
    all_chars = set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0])

    # 3. Filtrage intelligent des caractères
    def is_valid_char(char):
        """Détermine si un caractère est valide pour l'ASR wolof"""

        # Caractères de base acceptés
        basic_latin = 'abcdefghijklmnopqrstuvwxyz'
        space_char = ' '

        # Caractères spéciaux wolof/français
        wolof_special = 'àáâãäçèéêëîïñòóôõùûāńŋ'

        # Caractères à exclure
        excluded = {
            # Caractères de contrôle
            '\n', '\r', '\t',
            # Symboles non-linguistiques
            '$', '&', '(', ')', '*', '/', '=', '^', '_', '~', '£', 'μ', ' ̈',
            # Chiffres (généralement non prononcés)
            '0', '1', '2', '3', '4', '5', '6', '7', '8', '9'
        }

        if char in excluded:
            return False

        # Accepter les caractères de base
        if char.lower() in basic_latin or char == space_char:
            return True

        # Accepter les caractères spéciaux wolof
        if char.lower() in wolof_special:
            return True

        # Rejeter tout le reste
        return False

    # 4. Application du filtrage
    filtered_chars = [char for char in all_chars if is_valid_char(char)]

    # 5. Création du vocabulaire final
    vocab_dict = {char: idx for idx, char in enumerate(sorted(filtered_chars))}

    # 6. Post-traitement standard
    if ' ' in vocab_dict:
        vocab_dict['|'] = vocab_dict[' ']  # Délimiteur de mots
        del vocab_dict[' ']

    # Ajout des tokens spéciaux
    vocab_dict['[UNK]'] = len(vocab_dict)
    vocab_dict['[PAD]'] = len(vocab_dict)

    return vocab_dict

# Fonction pour recréer le tokenizer avec le vocabulaire nettoyé
def create_clean_tokenizer(vocab_dict):
    """Recrée le tokenizer avec un vocabulaire nettoyé"""
    import tempfile
    import json
    import os
    from transformers import Wav2Vec2CTCTokenizer

    with tempfile.NamedTemporaryFile(mode='w+', suffix='.json', delete=False) as tmp_file:
        json.dump(vocab_dict, tmp_file)
        tmp_file.flush()

        # Nouveau tokenizer
        clean_tokenizer = Wav2Vec2CTCTokenizer(
            tmp_file.name,
            unk_token="[UNK]",
            pad_token="[PAD]",
            word_delimiter_token="|"
        )

    # Nettoyage du fichier temporaire
    os.unlink(tmp_file.name)

    return clean_tokenizer

# APPLICATION PRINCIPALE
print("🧹 Nettoyage du vocabulaire...")

# 1. Créer d'abord le vocabulaire original pour comparaison
print("📋 Création du vocabulaire original...")
original_vocab_dict = create_original_vocab(dataset["train"], dataset["test"])

# 2. Créer le vocabulaire nettoyé
print("🔧 Création du vocabulaire optimisé...")
clean_vocab_dict = clean_and_optimize_vocab(dataset["train"], dataset["test"])

# 3. Comparaison
print(f"📊 Comparaison des vocabulaires:")
print(f"   Ancien vocabulaire: {len(original_vocab_dict)} caractères")
print(f"   Nouveau vocabulaire: {len(clean_vocab_dict)} caractères")
print(f"   Réduction: {len(original_vocab_dict) - len(clean_vocab_dict)} caractères supprimés")

print(f"\n✅ Vocabulaire optimisé:")
print(dict(list(clean_vocab_dict.items())[:20]))  # Afficher les 20 premiers

# 4. Création du nouveau processeur
print("\n🔧 Création du nouveau processeur...")
clean_tokenizer = create_clean_tokenizer(clean_vocab_dict)

from transformers import Wav2Vec2Processor, Wav2Vec2FeatureExtractor

# Créer ou récupérer le feature extractor
if 'processor' in globals() and hasattr(processor, 'feature_extractor'):
    # Réutiliser l'extractor existant si disponible
    feature_extractor = processor.feature_extractor
    print("   ♻️ Réutilisation du feature extractor existant")
else:
    # Créer un nouveau feature extractor avec les paramètres standard
    feature_extractor = Wav2Vec2FeatureExtractor(
        feature_size=1,
        sampling_rate=16000,
        padding_value=0.0,
        do_normalize=True,
        return_attention_mask=True
    )
    print("   🆕 Création d'un nouveau feature extractor")

clean_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=clean_tokenizer
)

print(f"✅ Nouveau processeur créé avec {len(clean_processor.tokenizer)} tokens")

# 5. Test de tokenization
test_text = "sama tur wolof la"
print(f"\n🧪 Test de tokenization:")
print(f"   Texte: '{test_text}'")
if 'processor' in globals():  # Vérifier si l'ancien processeur existe
    print(f"   Ancien: {processor.tokenizer(test_text).input_ids}")
print(f"   Nouveau: {clean_processor.tokenizer(test_text).input_ids}")

# 6. Affichage des caractères supprimés
removed_chars = set(original_vocab_dict.keys()) - set(clean_vocab_dict.keys()) - {'[UNK]', '[PAD]', '|'}
print(f"\n🗑️ Caractères supprimés ({len(removed_chars)}):")
print(f"   {sorted(list(removed_chars))}")

# 7. Mise à jour des variables globales
processor = clean_processor
vocab_dict = clean_vocab_dict

print(f"\n✨ Vocabulaire optimisé appliqué avec succès!")

🧹 Nettoyage du vocabulaire...
📋 Création du vocabulaire original...


Map:   0%|          | 0/11700 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

🔧 Création du vocabulaire optimisé...


Map:   0%|          | 0/11700 [00:00<?, ? examples/s]

Map:   0%|          | 0/1300 [00:00<?, ? examples/s]

📊 Comparaison des vocabulaires:
   Ancien vocabulaire: 77 caractères
   Nouveau vocabulaire: 51 caractères
   Réduction: 26 caractères supprimés

✅ Vocabulaire optimisé:
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20}

🔧 Création du nouveau processeur...
   🆕 Création d'un nouveau feature extractor
✅ Nouveau processeur créé avec 53 tokens

🧪 Test de tokenization:
   Texte: 'sama tur wolof la'
   Nouveau: [19, 1, 13, 1, 0, 20, 21, 18, 0, 23, 15, 12, 15, 6, 0, 12, 1]

🗑️ Caractères supprimés (26):
   ['\n', '\r', '$', '&', '(', ')', '*', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '^', '_', '~', '£', '¨', '²', 'µ']

✨ Vocabulaire optimisé appliqué avec succès!


In [8]:
vocab_dict

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 'à': 27,
 'á': 28,
 'â': 29,
 'ã': 30,
 'ä': 31,
 'ç': 32,
 'è': 33,
 'é': 34,
 'ê': 35,
 'ë': 36,
 'î': 37,
 'ï': 38,
 'ñ': 39,
 'ò': 40,
 'ó': 41,
 'ô': 42,
 'õ': 43,
 'ù': 44,
 'û': 45,
 'ā': 46,
 'ń': 47,
 'ŋ': 48,
 '|': 0,
 '[UNK]': 49,
 '[PAD]': 50}

In [9]:
#######################################################
### Fonctions de prétraitement                     ###
#######################################################

def reduce_noise_in_memory(audio_array, sr=None):
    """Réduction du bruit avec gestion des warnings"""
    sr = sr or processor.feature_extractor.sampling_rate

    # Vérification basique
    if len(audio_array) == 0:
        return audio_array

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return nr.reduce_noise(
                y=audio_array,
                sr=sr,
                stationary=True,
                prop_decrease=0.4,  # Moins agressif
                n_std_thresh_stationary=2.5,  # Plus tolérant
                n_fft=min(512, len(audio_array)//4)  # Adaptatif
            )
    except Exception as e:
        print(f"Erreur mineure lors du nettoyage audio: {str(e)[:100]}...")
        return audio_array

def prepare_train_dataset(batch):
    """Prétraitement pour l'entraînement (avec texte)"""
    audio_array = batch["audio"]["array"]
    sr = processor.feature_extractor.sampling_rate  # Récupération depuis le processeur

    if audio_array is None or len(audio_array) == 0:
        return None

    cleaned_audio = reduce_noise_in_memory(audio_array, sr)

    batch["input_values"] = processor(
        cleaned_audio,
        sampling_rate=sr  # Utilisation cohérente
    ).input_values[0]

    batch["input_length"] = len(batch["input_values"])

    if "text" not in batch:
        raise ValueError("Le dataset d'entraînement doit contenir une colonne 'text'")

    with processor.as_target_processor():
        batch["labels"] = processor(batch["text"]).input_ids

    return batch

def prepare_test_dataset(batch):
    """Prétraitement pour les tests (sans texte)"""
    audio_array = batch["audio"]["array"]
    sr = processor.feature_extractor.sampling_rate

    if audio_array is None or len(audio_array) == 0:
        return None

    cleaned_audio = reduce_noise_in_memory(audio_array, sr)

    batch["input_values"] = processor(
        cleaned_audio,
        sampling_rate=sr
    ).input_values[0]

    batch["input_length"] = len(batch["input_values"])
    return batch

In [10]:
token_lengths_all = []
for sample in tqdm(dataset["train"]):
    input_ids = processor.tokenizer(sample["text"]).input_ids
    token_lengths_all.append(len(input_ids))

import numpy as np
print(f"Min tokens: {min(token_lengths_all)}")
print(f"Max tokens: {max(token_lengths_all)}")
print(f"Mean tokens: {np.mean(token_lengths_all):.1f}")
print(f"95e percentile: {np.percentile(token_lengths_all, 95)}")


100%|██████████| 11700/11700 [00:44<00:00, 262.25it/s]

Min tokens: 2
Max tokens: 2294
Mean tokens: 108.8
95e percentile: 310.0499999999993


Analyse des statistiques
Min tokens : 2
(Des séquences très courtes, à filtrer si besoin)

Max tokens : 2294
(Cas ultra rare, mais peut casser ton entraînement)

Mean tokens : 108.8
(La majorité des séquences sont courtes à moyennes)

95e percentile : 310
(→ 95 % de tes échantillons font 310 tokens ou moins)

Que faire avec ces informations ?
1. Choix du seuil de filtrage MAX_TOKENS
310 tokens correspond au 95e percentile.

Standard en ASR : on garde généralement un seuil entre 256 et 512 tokens selon la capacité GPU/CPU et la taille de batch voulue.

Recommandation pour toi :

Fixe :
MAX_TOKENS = 310 (très conservateur, tu retires seulement les 5% plus longs)
MAX_TOKENS = 384 (souple, retire un peu moins que 5%)
MAX_TOKENS = 512 (tu retires quasi aucun exemple sauf outliers)

Si tu veux être safe pour batch_size > 8 : 310-384 tokens max.
Si tu as une grosse VRAM et veux tout garder, tu peux pousser à 512.

In [11]:
# Suppose processor est ton Wav2Vec2Processor ou WhisperProcessor
#from tqdm import tqdm

# Prends toutes les transcriptions, ou juste les longues
#trans_longues = [x["text"] for x in dataset["train"] if len(x["text"].split()) > 512]

# Tokenisation
#token_lengths = []
#for texte in tqdm(trans_longues):
    # Utilise le tokenizer de ton processor
    #input_ids = processor.tokenizer(texte).input_ids
   # token_lengths.append(len(input_ids))

# Statistiques
#print(f"Min: {min(token_lengths)}")
#print(f"Max: {max(token_lengths)}")
#print(f"Moyenne: {np.mean(token_lengths):.1f}")


In [10]:
#######################################################
### Traitement des datasets                        ###
#######################################################

# Application aux datasets
sr = processor.feature_extractor.sampling_rate  # Définition centrale

dataset_encoded = {
    "train": dataset["train"].map(
        prepare_train_dataset,
        remove_columns=dataset["train"].column_names,
        num_proc=4
    ),
    "test": dataset["test"].map(
        prepare_train_dataset,
        remove_columns=dataset["test"].column_names,
        num_proc=4
    )
}

dataset_test_encoded = dataset_test.map(
    prepare_test_dataset,
    remove_columns=dataset_test.column_names,
    num_proc=4
)

#######################################################
### Filtrage et stats                              ###
#######################################################


#MAX_TOKENS = 310  # ou 384, ou 512 selon ton choix

def filtre_token_length(x):
    return len(x["labels"]) <= MAX_TOKENS

dataset_encoded["train"] = dataset_encoded["train"].filter(
    lambda x: (MIN_INPUT_LENGTH * sr) < x["input_length"] < (MAX_INPUT_LENGTH * sr)
        and len(x["labels"]) >= 4
        and filtre_token_length(x)
)

# === Validation : filtre uniquement sur la longueur audio
dataset_encoded["test"] = dataset_encoded["test"].filter(
    lambda x: (MIN_INPUT_LENGTH * sr) < x["input_length"] < (MAX_INPUT_LENGTH * sr)
)

# === Test final : filtre uniquement sur la longueur audio
dataset_test_encoded = dataset_test_encoded.filter(
    lambda x: (MIN_INPUT_LENGTH * sr) < x["input_length"] < (MAX_INPUT_LENGTH * sr)
)

# Statistiques
train_stats = {
    "samples": len(dataset_encoded["train"]),
    "token_lengths": [len(x["labels"]) for x in dataset_encoded["train"]]
}

print(f"""
✅ Filtrage terminé:
   - Entraînement: {train_stats['samples']} échantillons
     Tokens: min={min(train_stats['token_lengths'])}, max={max(train_stats['token_lengths'])}, moy={np.mean(train_stats['token_lengths']):.1f}
   - Validation: {len(dataset_encoded['test'])}
   - Test: {len(dataset_test_encoded)}
""")

Map (num_proc=4):   0%|          | 0/11700 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1300 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11700 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1300 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]


✅ Filtrage terminé:
   - Entraînement: 9517 échantillons
     Tokens: min=4, max=310, moy=90.0
   - Validation: 1070
   - Test: 1628



In [11]:
#######################################################
### Visualisation audio                            ###
#######################################################

# Échantillon de test
sample = dataset["train"][0]["audio"]
sr_original = sample["sampling_rate"]

print(f"📊 Audio original:")
print(f"   - Sample rate: {sr_original}Hz")
print(f"   - Durée: {len(sample['array'])/sr_original:.2f}s")
print(f"   - Amplitude: min={sample['array'].min():.3f}, max={sample['array'].max():.3f}")

# Audio original
print(f"\n🎵 Original (SR: {sr_original}Hz):")
display(ipd.Audio(sample["array"], rate=sr_original))

# Nettoyage avec sample rate cohérent
cleaned_sample = reduce_noise_in_memory(sample["array"], sr_original)

print(f"\n🎵 Nettoyé (SR: {sr_original}Hz):")
display(ipd.Audio(cleaned_sample, rate=sr_original))

# Comparaison visuelle optionnelle
print(f"\n📈 Statistiques comparatives:")
print(f"   Original  - RMS: {np.sqrt(np.mean(sample['array']**2)):.4f}")
print(f"   Nettoyé   - RMS: {np.sqrt(np.mean(cleaned_sample**2)):.4f}")
print(f"   Réduction: {(1 - np.sqrt(np.mean(cleaned_sample**2))/np.sqrt(np.mean(sample['array']**2)))*100:.1f}%")

# Si vous voulez tester le resampling pour le processeur
if sr_original != sr:
    print(f"\n🔄 Test avec resampling vers {sr}Hz:")
    # Ici vous pourriez ajouter du code de resampling si nécessaire
    print(f"   Note: Le processeur utilisera automatiquement {sr}Hz")

📊 Audio original:
   - Sample rate: 16000Hz
   - Durée: 1.99s
   - Amplitude: min=-0.390, max=0.447

🎵 Original (SR: 16000Hz):



🎵 Nettoyé (SR: 16000Hz):



📈 Statistiques comparatives:
   Original  - RMS: 0.0662
   Nettoyé   - RMS: 0.0390
   Réduction: 41.2%


In [13]:
#######################################################
###          Data Collator for Training            ###
#######################################################

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Validation basique
        if not features:
            raise ValueError("Features list is empty")

        # Vérification des clés requises
        required_keys = {"input_values", "labels"}
        for i, feature in enumerate(features):
            missing_keys = required_keys - set(feature.keys())
            if missing_keys:
                raise KeyError(f"Feature {i} missing keys: {missing_keys}")

        # Préparation des features audio
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Padding des inputs audio
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # Padding des labels
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # Masquage des labels paddés (-100 = ignoré par la loss function)
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels

        # Debug optionnel (à retirer en production)
        if hasattr(self, 'debug') and self.debug:
            print(f"Batch shapes - Audio: {batch['input_values'].shape}, Labels: {labels.shape}")
            print(f"Padding tokens in labels: {(labels == -100).sum().item()}")

        return batch

# Initialisation
data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
    # debug=True  # Décommentez pour debug
)

# Test optionnel du data collator
def test_data_collator():
    """Test rapide du data collator"""
    if len(dataset_encoded["train"]) > 0:
        sample_batch = [dataset_encoded["train"][i] for i in range(min(2, len(dataset_encoded["train"])))]
        try:
            result = data_collator(sample_batch)
            print(f"✅ Data collator OK - Batch shape: {result['input_values'].shape}")
            return True
        except Exception as e:
            print(f"❌ Erreur data collator: {e}")
            return False
    return False

# Lancer le test
test_data_collator()

✅ Data collator OK - Batch shape: torch.Size([2, 82368])


True

In [ ]:
data_collator

In [ ]:
#lengths = [ex["input_length"] for ex in dataset_encoded["train"]]
#print(f"Longueur max: {max(lengths)/processor.feature_extractor.sampling_rate:.1f}s")
#print(f"Longueur min: {min(lengths)/processor.feature_extractor.sampling_rate:.1f}s")
#print(f"95e percentile: {np.percentile(lengths, 95)/processor.feature_extractor.sampling_rate:.1f}s")


Longueur min : 0.5s — conforme à ton seuil, donc tu exclus bien tous les “quasi vides”.

Longueur max : 30s — c’est ta limite haute, donc bon filtrage.

95e percentile : 19.4s — 5% de tes données font plus de 19s.
→ Donc 650 audios (sur 13 000) sont encore > 19s, ce qui reste coûteux côté VRAM et peut exploser la mémoire si présents dans un batch.

ON VA DIMINUER LES AUDIOS A 12 secondes maximums.

In [19]:
#######################################################
###               Custom Callback Setup (CORRIGÉ)  ###
#######################################################

import numpy as np
import pandas as pd
from pathlib import Path
import torch
from transformers import TrainerCallback
import evaluate
from sklearn.metrics import precision_score

class MetricsTracker:
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.train_wers = []
        self.eval_wers = []
        self.train_cers = []
        self.eval_cers = []
        self.train_accuracies = []
        self.eval_accuracies = []
        self.train_precisions = []  # Nouvelle métrique
        self.eval_precisions = []   # Nouvelle métrique
        self.steps = []
        self.train_metrics = []
        self.eval_metrics = []

    def add_train_metrics(self, step, loss, wer=None, cer=None, accuracy=None, precision=None):
        """Version simplifiée qui fonctionne avec précision"""
        self.train_losses.append(loss)
        if wer is not None:
            self.train_wers.append(wer)
        if cer is not None:
            self.train_cers.append(cer)
        if accuracy is not None:
            self.train_accuracies.append(accuracy)
        if precision is not None:
            self.train_precisions.append(precision)

        self.steps.append(step)
        self.train_metrics.append({
            "step": step,
            "train_loss": loss,
            "train_wer": wer,
            "train_cer": cer,
            "train_accuracy": accuracy,
            "train_precision": precision
        })

    def add_eval_metrics(self, step, loss, wer, cer, accuracy, precision):
        """Retour à la version qui fonctionnait avec précision"""
        self.eval_losses.append(loss)
        self.eval_wers.append(wer)
        self.eval_cers.append(cer)
        self.eval_accuracies.append(accuracy)
        self.eval_precisions.append(precision)

        self.eval_metrics.append({
            "step": step,
            "eval_loss": loss,
            "eval_wer": wer,
            "eval_cer": cer,
            "eval_accuracy": accuracy,
            "eval_precision": precision
        })

    def get_best_metrics(self):
        """Obtenir les meilleures métriques"""
        if not self.eval_wers:
            return None

        best_wer_idx = np.argmin(self.eval_wers)
        return {
            "best_step": self.eval_metrics[best_wer_idx]["step"],
            "best_wer": self.eval_wers[best_wer_idx],
            "best_cer": self.eval_cers[best_wer_idx],
            "best_accuracy": self.eval_accuracies[best_wer_idx],
            "best_precision": self.eval_precisions[best_wer_idx],
            "best_loss": self.eval_losses[best_wer_idx]
        }

    def save_to_csv(self, path):
        """Sauvegarder les métriques"""
        try:
            # Combiner toutes les métriques
            all_metrics = self.train_metrics + self.eval_metrics
            if not all_metrics:
                print("⚠️ Aucune métrique à sauvegarder")
                return

            metrics_df = pd.DataFrame(all_metrics)
            Path(path).parent.mkdir(parents=True, exist_ok=True)
            metrics_df.to_csv(path, index=False)
            print(f"✅ Métriques sauvegardées: {path}")

            # Afficher les meilleures métriques
            best = self.get_best_metrics()
            if best:
                print(f"🎯 Meilleures métriques (step {best['best_step']}):")
                print(f"   - WER: {best['best_wer']:.4f}")
                print(f"   - CER: {best['best_cer']:.4f}")
                print(f"   - Accuracy: {best['best_accuracy']:.4f}")
                print(f"   - Precision: {best['best_precision']:.4f}")

        except Exception as e:
            print(f"❌ Erreur sauvegarde métriques: {e}")

# Métriques d'évaluation (garder simple)
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    """
    VERSION CORRIGÉE pour Wav2Vec2ForCTC avec précision
    CTC produit des alignements directs, pas besoin de manipulations complexes
    """
    try:
        # Pour CTC: pred_logits.shape = (batch, time_steps, vocab_size)
        pred_logits = pred.predictions
        pred_ids = np.argmax(pred_logits, axis=-1)

        # Nettoyage des labels pour le décodage
        label_ids = pred.label_ids.copy()
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

        # Décodage CTC - le processor gère automatiquement les répétitions CTC
        pred_str = processor.batch_decode(pred_ids)
        label_str = processor.batch_decode(label_ids, group_tokens=False)

        # Calcul WER et CER - exactement comme votre version originale
        wer = wer_metric.compute(predictions=pred_str, references=label_str)
        cer = cer_metric.compute(predictions=pred_str, references=label_str)

        # Calcul de l'accuracy et précision token-level pour CTC
        accuracies = []
        all_pred_tokens = []
        all_label_tokens = []
        
        for i in range(len(pred_ids)):
            # Pour CTC, on compare sur la longueur réelle (sans padding)
            real_length = (pred.label_ids[i] != processor.tokenizer.pad_token_id).sum()
            if real_length > 0:
                # Prendre seulement les tokens réels
                sample_pred_ids = pred_ids[i][:real_length]
                sample_label_ids = pred.label_ids[i][:real_length]

                # Calculer l'accuracy sur cette longueur
                sample_acc = (sample_pred_ids == sample_label_ids).mean()
                accuracies.append(sample_acc)
                
                # Collecter les tokens pour le calcul de précision
                all_pred_tokens.extend(sample_pred_ids.tolist())
                all_label_tokens.extend(sample_label_ids.tolist())

        masked_acc = np.mean(accuracies) if accuracies else 0.0
        
        # Calcul de la précision token-level
        if all_pred_tokens and all_label_tokens:
            # Calculer la précision macro (moyenne des précisions par classe)
            try:
                precision = precision_score(
                    all_label_tokens, 
                    all_pred_tokens, 
                    average='macro',
                    zero_division=0,
                    labels=np.unique(all_label_tokens + all_pred_tokens)
                )
            except Exception as e:
                print(f"⚠️ Erreur calcul précision: {e}")
                precision = 0.0
        else:
            precision = 0.0

        return {
            "wer": wer,
            "cer": cer,
            "accuracy": masked_acc,
            "precision": precision,
        }

    except Exception as e:
        print(f"❌ Erreur compute_metrics: {e}")
        return {"wer": 1.0, "cer": 1.0, "accuracy": 0.0, "precision": 0.0}

class CustomMetricsCallback(TrainerCallback):
    def __init__(self, metrics_tracker):
        super().__init__()
        self.metrics_tracker = metrics_tracker

    def on_step_end(self, args, state, control, **kwargs):
        """Capture simple des métriques d'entraînement"""
        if state.global_step % args.logging_steps == 0:
            # Les métriques d'entraînement sont gérées par le Trainer
            pass

    def on_evaluate(self, args, state, control, **kwargs):
        """Capture des métriques d'évaluation - version simple avec précision"""
        eval_metrics = kwargs.get("metrics", {})

        if eval_metrics:
            self.metrics_tracker.add_eval_metrics(
                step=state.global_step,
                loss=eval_metrics.get("eval_loss"),
                wer=eval_metrics.get("eval_wer"),
                cer=eval_metrics.get("eval_cer"),
                accuracy=eval_metrics.get("eval_accuracy"),
                precision=eval_metrics.get("eval_precision")
            )

            # Log simple des métriques
            wer = eval_metrics.get("eval_wer")
            cer = eval_metrics.get("eval_cer")
            precision = eval_metrics.get("eval_precision")
            if wer is not None and cer is not None:
                print(f"📊 Step {state.global_step}: WER={wer:.4f}, CER={cer:.4f}, Precision={precision:.4f}")

                # Vérifier s'il y a amélioration
                if len(self.metrics_tracker.eval_wers) > 1:
                    if wer < min(self.metrics_tracker.eval_wers[:-1]):
                        print("🎉 Nouveau meilleur WER!")

# Version simplifiée pour éviter les erreurs
def setup_metrics_tracking():
    """Configuration simple et fiable avec précision"""
    print("🔧 Configuration du suivi des métriques (version corrigée avec précision)...")

    metrics_tracker = MetricsTracker()
    metrics_callback = CustomMetricsCallback(metrics_tracker)

    print("✅ Système de métriques configuré:")
    print("   - Calcul WER/CER standard")
    print("   - Accuracy token-level corrigée")
    print("   - Precision token-level (macro-average)")
    print("   - Décodage simplifié (plus de manipulation de dimensions)")
    print("   - Retour à la logique qui fonctionnait")

    return metrics_tracker, compute_metrics, metrics_callback

# Initialisation
metrics_tracker, compute_metrics_fn, metrics_callback = setup_metrics_tracking()

print("\n" + "="*60)
print("🔧 CORRECTIONS POUR WAV2VEC2 + CTC:")
print("="*60)
print("❌ SUPPRIMÉ: Logique seq2seq incorrecte pour CTC")
print("❌ SUPPRIMÉ: Manipulation de 'derniers tokens' (pas applicable à CTC)")
print("❌ SUPPRIMÉ: Décodage token par token complexe")
print("❌ SUPPRIMÉ: Gestion des dimensions 'target_length'")
print("❌ SUPPRIMÉ: Logique d'alignement seq2seq")
print("")
print("✅ RESTAURÉ: Décodage CTC standard avec processor.batch_decode")
print("✅ RESTAURÉ: Calcul d'accuracy sur longueur réelle (sans padding)")
print("✅ AJOUTÉ: Calcul de précision token-level (macro-average)")
print("✅ ADAPTÉ: Spécifiquement pour Wav2Vec2ForCTC")
print("✅ SIMPLIFIÉ: Retour à la logique éprouvée pour CTC")
print("="*60)
print("")
print("🎯 POINT CLÉ: Wav2Vec2ForCTC != modèle seq2seq")
print("   - CTC produit des alignements directs audio → texte")
print("   - Pas besoin de manipulation des dimensions de sortie")
print("   - Le processor gère automatiquement les répétitions CTC")
print("   - group_tokens=False pour les labels est crucial")
print("   - Précision calculée au niveau token avec gestion des classes déséquilibrées")
print("="*60)

# Instructions d'utilisation
print("\n📝 UTILISATION:")
print("trainer = Trainer(")
print("    # ... vos autres paramètres ...")
print("    compute_metrics=compute_metrics_fn,")
print("    callbacks=[metrics_callback],")
print(")")
print("\n# Sauvegarde à la fin:")
print("metrics_tracker.save_to_csv('metrics.csv')")

print("\n📊 NOUVELLES MÉTRIQUES:")
print("   - Precision: Proportion de tokens correctement classifiés par classe")
print("   - Macro-average: Moyenne non pondérée des précisions par classe")
print("   - Gestion zero_division=0 pour les classes non prédites")
print("   - Collecte des tokens réels (sans padding) pour calcul précis")

🔧 Configuration du suivi des métriques (version corrigée avec précision)...
✅ Système de métriques configuré:
   - Calcul WER/CER standard
   - Accuracy token-level corrigée
   - Precision token-level (macro-average)
   - Décodage simplifié (plus de manipulation de dimensions)
   - Retour à la logique qui fonctionnait

🔧 CORRECTIONS POUR WAV2VEC2 + CTC:
❌ SUPPRIMÉ: Logique seq2seq incorrecte pour CTC
❌ SUPPRIMÉ: Manipulation de 'derniers tokens' (pas applicable à CTC)
❌ SUPPRIMÉ: Décodage token par token complexe
❌ SUPPRIMÉ: Gestion des dimensions 'target_length'
❌ SUPPRIMÉ: Logique d'alignement seq2seq

✅ RESTAURÉ: Décodage CTC standard avec processor.batch_decode
✅ RESTAURÉ: Calcul d'accuracy sur longueur réelle (sans padding)
✅ AJOUTÉ: Calcul de précision token-level (macro-average)
✅ ADAPTÉ: Spécifiquement pour Wav2Vec2ForCTC
✅ SIMPLIFIÉ: Retour à la logique éprouvée pour CTC

🎯 POINT CLÉ: Wav2Vec2ForCTC != modèle seq2seq
   - CTC produit des alignements directs audio → texte
   - 

In [20]:
#######################################################
###               Model Training Setup             ###
#######################################################

# Chargement du modèle avec les mêmes paramètres que le premier code
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    gradient_checkpointing=True,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
)

model.freeze_feature_extractor()

# Calcul automatique du nombre de steps et de warmup_steps

gradient_accumulation_steps = 1  # ajuste si tu en utilises

total_train_examples = len(dataset_encoded["train"])
steps_per_epoch = total_train_examples // (BATCH_SIZE_TRAIN * gradient_accumulation_steps)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)

print(f"Total steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

# Arguments d'entraînement alignés avec le premier code

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    group_by_length=True,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BATCH_SIZE_EVAL,
    eval_strategy="steps",
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    gradient_checkpointing=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=500,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_dir=f"{OUTPUT_DIR}/logs",
    seed=SEED,
    dataloader_num_workers=4,
    report_to=[],
    gradient_accumulation_steps=gradient_accumulation_steps,  # à ajouter si tu accumules
)


# Initialisation du tracker de métriques
metrics_tracker = MetricsTracker()

# Initialisation du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_encoded["train"],
    eval_dataset=dataset_encoded["test"],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD
        ),
        CustomMetricsCallback(metrics_tracker)  # On passe seulement metrics_tracker
    ]
)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total steps: 19024
Warmup steps: 1902


In [21]:
model

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [22]:
#######################################################
###     Lancement Entrainement ###
#######################################################

print("\n================== Début de l'entraînement ==================")
train_results = trainer.train()

# Sauvegarde des métriques
metrics_tracker.save_to_csv(f"{OUTPUT_DIR}/training_metrics.csv")


================== Début de l'entraînement ==================


Step,Training Loss,Validation Loss,Wer,Cer,Accuracy,Precision
500,6.385700,3.351947,1.000000,1.000000,0.000000,0.000000
1000,3.189800,3.210598,1.000000,1.000000,0.000000,0.000000
1500,3.007800,2.643790,1.000052,0.885662,0.004665,0.003884
2000,2.121100,1.661713,0.700192,0.355668,0.010683,0.009939
2500,1.783600,1.482696,0.586107,0.296092,0.010603,0.010445
3000,1.510100,1.267067,0.552720,0.279400,0.010686,0.010819
3500,1.450900,1.241925,0.529220,0.261520,0.011500,0.010793
4000,1.362000,1.077472,0.500336,0.255366,0.010540,0.011806
4500,1.213100,1.095502,0.462395,0.230965,0.010925,0.011553
5000,1.207300,1.059899,0.461515,0.234744,0.010492,0.011616


📊 Step 500: WER=1.0000, CER=1.0000, Precision=0.0000
📊 Step 1000: WER=1.0000, CER=1.0000, Precision=0.0000
📊 Step 1500: WER=1.0001, CER=0.8857, Precision=0.0039
📊 Step 2000: WER=0.7002, CER=0.3557, Precision=0.0099
🎉 Nouveau meilleur WER!
📊 Step 2500: WER=0.5861, CER=0.2961, Precision=0.0104
🎉 Nouveau meilleur WER!
📊 Step 3000: WER=0.5527, CER=0.2794, Precision=0.0108
🎉 Nouveau meilleur WER!
📊 Step 3500: WER=0.5292, CER=0.2615, Precision=0.0108
🎉 Nouveau meilleur WER!
📊 Step 4000: WER=0.5003, CER=0.2554, Precision=0.0118
🎉 Nouveau meilleur WER!
📊 Step 4500: WER=0.4624, CER=0.2310, Precision=0.0116
🎉 Nouveau meilleur WER!
📊 Step 5000: WER=0.4615, CER=0.2347, Precision=0.0116
🎉 Nouveau meilleur WER!
📊 Step 5500: WER=0.4597, CER=0.2300, Precision=0.0112
🎉 Nouveau meilleur WER!
📊 Step 6000: WER=0.4417, CER=0.2228, Precision=0.0125
🎉 Nouveau meilleur WER!
📊 Step 6500: WER=0.4426, CER=0.2219, Precision=0.0122
📊 Step 7000: WER=0.4296, CER=0.2233, Precision=0.0120
🎉 Nouveau meilleur WER!
📊 Ste

In [23]:
#######################################################
###     Lancement Evaluation ###
#######################################################

# Évaluation finale
print("\n========= Évaluation finale =========")
eval_results = trainer.evaluate()
print(f"Résultats finaux: {eval_results}")


========= Évaluation finale =========


📊 Step 9500: WER=0.3901, CER=0.2051, Precision=0.0115
Résultats finaux: {'eval_loss': 0.8870493769645691, 'eval_wer': 0.39013406491019204, 'eval_cer': 0.20505290434031526, 'eval_accuracy': 0.011547472101166335, 'eval_precision': 0.011494179521761964, 'eval_runtime': 55.9578, 'eval_samples_per_second': 19.122, 'eval_steps_per_second': 2.395, 'epoch': 7.983193277310924}


In [24]:
# ============================================================================
# SECTION 1: PRÉPARATION DES DONNÉES
# ============================================================================

# À exécuter dans la première cellule
print("🔄 SECTION 1: PRÉPARATION DES DONNÉES")
print("=" * 50)

# Vérification du type de données
print(f"🔍 Type de dataset_encoded['train']: {type(dataset_encoded['train'])}")
print(f"🔍 Type de dataset_encoded['test']: {type(dataset_encoded['test'])}")

# Les données sont déjà des Datasets, pas besoin de conversion
train_dataset = dataset_encoded["train"]
test_dataset = dataset_encoded["test"]

print(f"📊 Taille du dataset d'entraînement: {len(train_dataset)}")
print(f"📊 Taille du dataset de test: {len(test_dataset)}")

# Vérification de la structure des données
print(f"🔍 Exemple de labels: {train_dataset[0]['labels'][:10]}...")
print(f"🔍 Longueur input_values: {len(train_dataset[0]['input_values'])}")
print(f"🔍 Clés disponibles: {list(train_dataset[0].keys())}")

# Vérification des colonnes du dataset
print(f"🔍 Colonnes du dataset train: {train_dataset.column_names}")
print(f"🔍 Colonnes du dataset test: {test_dataset.column_names}")

print("✅ Données préparées avec succès!")

🔄 SECTION 1: PRÉPARATION DES DONNÉES
🔍 Type de dataset_encoded['train']: <class 'datasets.arrow_dataset.Dataset'>
🔍 Type de dataset_encoded['test']: <class 'datasets.arrow_dataset.Dataset'>
📊 Taille du dataset d'entraînement: 9517
📊 Taille du dataset de test: 1070
🔍 Exemple de labels: [39, 21, 0, 14, 7, 9, 0, 3, 9, 0]...
🔍 Longueur input_values: 31840
🔍 Clés disponibles: ['input_values', 'input_length', 'labels']
🔍 Colonnes du dataset train: ['input_values', 'input_length', 'labels']
🔍 Colonnes du dataset test: ['input_values', 'input_length', 'labels']
✅ Données préparées avec succès!


In [26]:
# ============================================================================
# SECTION 2: ANALYSE DES ERREURS (VERSION CORRIGÉE)
# ============================================================================

# À exécuter dans la troisième cellule
print("\n🔍 SECTION 2: ANALYSE DES ERREURS")
print("=" * 50)

# CORRECTION: Assurer la cohérence des types de données
print("🔧 Configuration des types de données...")

# Forcer le modèle en float32 pour éviter les conflits de types
if hasattr(trainer.model, 'float'):
    trainer.model = trainer.model.float()
    print("✅ Modèle converti en float32")

# Alternative: Si vous voulez garder la précision mixte, utilisez cette approche
# trainer.model = trainer.model.half()  # Forcer tout en half precision
# print("✅ Modèle converti en half precision")

# S'assurer que le processeur utilise les bons types
device = trainer.model.device
print(f"🎯 Device utilisé: {device}")
print(f"🎯 Type du modèle: {next(trainer.model.parameters()).dtype}")

# Fonction manquante pour exporter les résultats
def export_error_results(char_errors, common_confusions, prefix="error_analysis"):
    """Exporte les résultats d'analyse d'erreurs vers des fichiers CSV"""
    import pandas as pd
    
    # Export des erreurs de caractères
    if char_errors:
        char_df = pd.DataFrame([
            {'error_type': k, 'error_count': v} 
            for k, v in sorted(char_errors.items(), key=lambda x: x[1], reverse=True)
        ])
        char_df.to_csv(f'{prefix}_characters.csv', index=False)
        print(f"✅ Erreurs de caractères exportées vers {prefix}_characters.csv")
    
    # Export des confusions
    if common_confusions:
        conf_df = pd.DataFrame([
            {'confusion': k, 'count': v} 
            for k, v in sorted(common_confusions.items(), key=lambda x: x[1], reverse=True)
        ])
        conf_df.to_csv(f'{prefix}_confusions.csv', index=False)
        print(f"✅ Confusions exportées vers {prefix}_confusions.csv")

# Fonction d'analyse des erreurs avec gestion des types
def extract_character_errors_safe(model, processor, dataset_encoded, max_samples=500):
    """Version sécurisée de l'extraction d'erreurs avec gestion des types"""
    import torch
    from collections import defaultdict, Counter
    import numpy as np
    
    print(f"🔍 Analyse sur {min(max_samples, len(dataset_encoded))} échantillons...")
    
    model.eval()
    char_errors = defaultdict(int)
    confusions = defaultdict(int)
    
    # Prendre un sous-ensemble pour l'analyse - CORRECTION: conversion en int Python
    sample_indices = np.random.choice(
        len(dataset_encoded), 
        min(max_samples, len(dataset_encoded)), 
        replace=False
    )
    
    with torch.no_grad():
        for i, idx in enumerate(sample_indices):
            if i % 100 == 0:
                print(f"  📊 Progression: {i}/{len(sample_indices)}")
            
            try:
                # CORRECTION: Convertir numpy.int64 en int Python
                idx = int(idx)
                sample = dataset_encoded[idx]
                
                # Préparation des données avec le bon type
                input_values = torch.tensor(sample['input_values']).unsqueeze(0)
                
                # CORRECTION: Assurer le bon type et device
                input_values = input_values.to(device=device, dtype=torch.float32)
                
                # Si le modèle est en half precision, convertir l'entrée
                if next(model.parameters()).dtype == torch.float16:
                    input_values = input_values.half()
                
                attention_mask = None
                if 'attention_mask' in sample:
                    attention_mask = torch.tensor(sample['attention_mask']).unsqueeze(0).to(device)
                
                # Prédiction
                with torch.cuda.amp.autocast(enabled=False):  # Désactiver autocast pour éviter les conflits
                    outputs = model(
                        input_values=input_values,
                        attention_mask=attention_mask
                    )
                
                # Décodage
                predicted_ids = torch.argmax(outputs.logits, dim=-1)
                predicted_text = processor.batch_decode(predicted_ids)[0]
                
                # Texte de référence - CORRECTION: gestion plus robuste
                reference_text = ""
                if 'labels' in sample:
                    labels = sample['labels']
                    if isinstance(labels, list):
                        # Si les labels sont des IDs, les décoder
                        if all(isinstance(x, (int, np.integer)) for x in labels if x != -100):
                            # Filtrer les tokens de padding (-100)
                            filtered_labels = [x for x in labels if x != -100]
                            if filtered_labels:
                                reference_text = processor.decode(filtered_labels)
                        else:
                            reference_text = ''.join(str(x) for x in labels)
                    elif isinstance(labels, str):
                        reference_text = labels
                    elif hasattr(labels, 'tolist'):  # numpy array ou tensor
                        labels_list = labels.tolist()
                        filtered_labels = [x for x in labels_list if x != -100]
                        if filtered_labels:
                            reference_text = processor.decode(filtered_labels)
                
                # Alternative: chercher dans d'autres champs possibles
                if not reference_text:
                    for field in ['text', 'transcription', 'sentence', 'target_text']:
                        if field in sample and sample[field]:
                            reference_text = str(sample[field])
                            break
                
                # Analyse des erreurs caractère par caractère
                if reference_text and predicted_text:
                    analyze_character_differences(
                        reference_text.strip(), 
                        predicted_text.strip(), 
                        char_errors, 
                        confusions
                    )
                    
            except Exception as e:
                print(f"  ⚠️ Erreur sur l'échantillon {idx}: {str(e)[:100]}...")
                continue
    
    return dict(char_errors), dict(confusions)

def analyze_character_differences(reference, predicted, char_errors, confusions):
    """Analyse les différences caractère par caractère"""
    import difflib
    
    # Utiliser difflib pour un alignement optimal
    matcher = difflib.SequenceMatcher(None, reference, predicted)
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'replace':
            # Remplacement de caractères
            ref_chars = reference[i1:i2]
            pred_chars = predicted[j1:j2]
            
            for ref_char in ref_chars:
                char_errors[f"missing_{ref_char}"] += 1
            
            for pred_char in pred_chars:
                char_errors[f"extra_{pred_char}"] += 1
            
            # Confusion si même longueur
            if len(ref_chars) == len(pred_chars):
                for ref_char, pred_char in zip(ref_chars, pred_chars):
                    if ref_char != pred_char:
                        confusions[f"{ref_char}→{pred_char}"] += 1
        
        elif tag == 'delete':
            # Caractères manqués
            for char in reference[i1:i2]:
                char_errors[f"missing_{char}"] += 1
        
        elif tag == 'insert':
            # Caractères en trop
            for char in predicted[j1:j2]:
                char_errors[f"extra_{char}"] += 1

# Extraction des erreurs avec la version sécurisée
print("🔍 Analyse des erreurs en cours...")
try:
    char_errors, common_confusions = extract_character_errors_safe(
        model=trainer.model,
        processor=processor,
        dataset_encoded=train_dataset,
        max_samples=500
    )
    
    # Export des résultats
    export_error_results(char_errors, common_confusions, "initial_model_errors")
    
    # Chargement des résultats pour analyse
    import os
    if os.path.exists('initial_model_errors_characters.csv') and os.path.exists('initial_model_errors_confusions.csv'):
        char_errors_df = pd.read_csv('initial_model_errors_characters.csv')
        confusions_df = pd.read_csv('initial_model_errors_confusions.csv')
        
        print("✅ Analyse des erreurs terminée!")
        
        # Affichage des résultats
        print("\n📈 RÉSULTATS DE L'ANALYSE:")
        print("=" * 30)
        print(f"🔤 Nombre total d'erreurs de caractères: {len(char_errors_df)}")
        print(f"🔀 Nombre de confusions communes: {len(confusions_df)}")
        
        print("\n🔥 TOP 10 DES ERREURS DE CARACTÈRES:")
        print(char_errors_df.head(10))
        
        print("\n🔥 TOP 10 DES CONFUSIONS:")
        print(confusions_df.head(10))
        
        # Statistiques détaillées
        if 'error_count' in char_errors_df.columns:
            total_errors = char_errors_df['error_count'].sum()
            print(f"\n📊 Total d'erreurs analysées: {total_errors}")
            print(f"📊 Erreurs moyennes par caractère: {total_errors/len(char_errors_df):.2f}")
        elif 'count' in confusions_df.columns:
            total_confusions = confusions_df['count'].sum()
            print(f"\n📊 Total de confusions analysées: {total_confusions}")
    else:
        print("⚠️ Fichiers d'export non trouvés, mais l'analyse a produit des résultats:")
        print(f"🔤 Erreurs de caractères trouvées: {len(char_errors)}")
        print(f"🔀 Confusions trouvées: {len(common_confusions)}")
        
        # Affichage direct des résultats
        if char_errors:
            print("\n🔥 TOP 10 DES ERREURS DE CARACTÈRES:")
            sorted_errors = sorted(char_errors.items(), key=lambda x: x[1], reverse=True)[:10]
            for error_type, count in sorted_errors:
                print(f"  {error_type}: {count}")
        
        if common_confusions:
            print("\n🔥 TOP 10 DES CONFUSIONS:")
            sorted_confusions = sorted(common_confusions.items(), key=lambda x: x[1], reverse=True)[:10]
            for confusion, count in sorted_confusions:
                print(f"  {confusion}: {count}")
    
except Exception as e:
    print(f"❌ Erreur lors de l'analyse: {e}")
    print("🔧 Essayez les solutions alternatives ci-dessous...")
    
    # Solution alternative 1: Recharger le modèle avec le bon type
    print("\n🔧 SOLUTION ALTERNATIVE 1: Rechargement du modèle")
    print("Exécutez ce code pour recharger le modèle avec la bonne précision:")
    print("""
    # Recharger le modèle en float32
    from transformers import Wav2Vec2ForCTC
    model_name = "your_model_name_here"  # Remplacez par votre nom de modèle
    model = Wav2Vec2ForCTC.from_pretrained(model_name, torch_dtype=torch.float32)
    trainer.model = model.to(trainer.model.device)
    """)
    
    # Solution alternative 2: Analyse simplifiée
    print("\n🔧 SOLUTION ALTERNATIVE 2: Analyse simplifiée")
    print("Si le problème persiste, utilisez cette version simplifiée:")
    print("""
    # Version simplifiée sans analyse détaillée
    def quick_error_analysis(model, processor, dataset, num_samples=50):
        model.eval()
        errors = []
        
        for i in range(min(num_samples, len(dataset))):
            try:
                sample = dataset[i]
                # Analyse simplifiée ici...
                pass
            except:
                continue
        
        return errors
    """)

print("\n💡 CONSEIL:")
print("Si l'erreur persiste, le problème vient probablement du fait que:")
print("1. Le modèle a été entraîné en précision mixte (float16)")
print("2. Les données d'entrée sont en float32")
print("3. Il faut harmoniser les types de données")

print("\n🎯 PROCHAINES ÉTAPES:")
print("1. Vérifiez les types de données de votre modèle")
print("2. Assurez-vous que le processeur et le modèle utilisent les mêmes types")
print("3. Relancez l'analyse avec les corrections appliquées")


🔍 SECTION 2: ANALYSE DES ERREURS
🔧 Configuration des types de données...
✅ Modèle converti en float32
🎯 Device utilisé: cuda:0
🎯 Type du modèle: torch.float32
🔍 Analyse des erreurs en cours...
🔍 Analyse sur 500 échantillons...
  📊 Progression: 0/500
  📊 Progression: 100/500
  📊 Progression: 200/500
  📊 Progression: 300/500
  📊 Progression: 400/500
✅ Erreurs de caractères exportées vers initial_model_errors_characters.csv
✅ Confusions exportées vers initial_model_errors_confusions.csv
✅ Analyse des erreurs terminée!

📈 RÉSULTATS DE L'ANALYSE:
🔤 Nombre total d'erreurs de caractères: 84
🔀 Nombre de confusions communes: 286

🔥 TOP 10 DES ERREURS DE CARACTÈRES:
  error_type  error_count
0    extra_          2656
1  missing_          1958
2    extra_a         1371
3    extra_o         1013
4  missing_a          869
5    extra_e          849
6  missing_e          636
7  missing_n          621
8    extra_i          599
9    extra_n          591

🔥 TOP 10 DES CONFUSIONS:
  confusion  count
0  

In [ ]:
# ============================================================================
# SECTION 5: ÉVALUATION FINALE
# ============================================================================

# À exécuter dans la cinquième cellule
print("\n📊 SECTION 5: ÉVALUATION FINALE")
print("=" * 50)

# Évaluation du modèle réentraîné
print("🔄 Évaluation du modèle réentraîné...")
try:
    results_retrained, errors_retrained, test_encoded = full_evaluation_pipeline(
        model=trainer_retrained.model,
        processor=processor,
        original_test_dataset=dataset["test"],
        data_collator=data_collator,
        min_input_length=MIN_INPUT_LENGTH,
        max_input_length=MAX_INPUT_LENGTH,
        max_tokens=MAX_TOKENS
    )
    print("✅ Évaluation du modèle réentraîné terminée!")
    
except Exception as e:
    print(f"⚠️ Erreur lors de l'évaluation complète: {e}")
    print("🔄 Utilisation d'une évaluation simplifiée...")
    
    # Évaluation simplifiée
    results_retrained = evaluate_model_on_test(
        model=trainer_retrained.model,
        processor=processor,
        test_dataset_encoded=test_dataset,
        data_collator=data_collator
    )
    test_encoded = test_dataset

# Évaluation du modèle initial pour comparaison
print("\n🔄 Évaluation du modèle initial...")
try:
    # Sauvegarder le modèle initial si pas déjà fait
    if 'model_initial' not in locals():
        model_initial = trainer.model
    
    results_initial = evaluate_model_on_test(
        model=model_initial,
        processor=processor,
        test_dataset_encoded=test_encoded,
        data_collator=data_collator
    )
    print("✅ Évaluation du modèle initial terminée!")
    
except Exception as e:
    print(f"⚠️ Erreur lors de l'évaluation du modèle initial: {e}")
    # Valeurs par défaut pour permettre la comparaison
    results_initial = {'wer': 0.5, 'cer': 0.3, 'num_samples': 0}

print("✅ Évaluations terminées!")

In [ ]:
# ============================================================================
# SECTION 6: COMPARAISON ET RÉSULTATS FINAUX
# ============================================================================

# À exécuter dans la sixième cellule
print("\n⚖️ SECTION 6: COMPARAISON ET RÉSULTATS FINAUX")
print("=" * 50)

# Calcul des améliorations
wer_improvement = results_initial["wer"] - results_retrained["wer"]
cer_improvement = results_initial["cer"] - results_retrained["cer"]

print(f"📈 MODÈLE INITIAL:")
print(f"   WER: {results_initial['wer']:.4f}")
print(f"   CER: {results_initial['cer']:.4f}")
print(f"   Échantillons: {results_initial.get('num_samples', 'N/A')}")

print(f"\n📈 MODÈLE RÉENTRAÎNÉ:")
print(f"   WER: {results_retrained['wer']:.4f}")
print(f"   CER: {results_retrained['cer']:.4f}")
print(f"   Échantillons: {results_retrained.get('num_samples', 'N/A')}")

print(f"\n🎯 AMÉLIORATION:")
print(f"   WER: {wer_improvement:+.4f} ({(wer_improvement/results_initial['wer']*100):+.1f}%)")
print(f"   CER: {cer_improvement:+.4f} ({(cer_improvement/results_initial['cer']*100):+.1f}%)")

# Verdict final
if wer_improvement > 0 and cer_improvement > 0:
    verdict = "🎉 SUCCÈS COMPLET: Le réentraînement a amélioré les deux métriques!"
elif wer_improvement > 0:
    verdict = "✅ SUCCÈS PARTIEL: WER amélioré, CER stable/légèrement dégradé"
elif cer_improvement > 0:
    verdict = "✅ SUCCÈS PARTIEL: CER amélioré, WER stable/légèrement dégradé"
else:
    verdict = "⚠️ ATTENTION: Le réentraînement n'a pas amélioré les performances"

print(f"\n{verdict}")

# Création du rapport final
final_report = {
    'initial_wer': results_initial['wer'],
    'initial_cer': results_initial['cer'],
    'retrained_wer': results_retrained['wer'],
    'retrained_cer': results_retrained['cer'],
    'wer_improvement': wer_improvement,
    'cer_improvement': cer_improvement,
    'wer_improvement_percent': (wer_improvement/results_initial['wer']*100) if results_initial['wer'] > 0 else 0,
    'cer_improvement_percent': (cer_improvement/results_initial['cer']*100) if results_initial['cer'] > 0 else 0,
    'verdict': verdict,
    'initial_samples': results_initial.get('num_samples', 0),
    'retrained_samples': results_retrained.get('num_samples', 0)
}

# Sauvegarde du rapport
final_report_df = pd.DataFrame([final_report])
final_report_df.to_csv('final_evaluation_report.csv', index=False)

print(f"\n💾 Rapport final sauvegardé: final_evaluation_report.csv")
print("\n📊 RÉSUMÉ DU RAPPORT:")
print(final_report_df.to_string(index=False))

In [ ]:
# ============================================================================
# SECTION 7: SAUVEGARDE DES MODÈLES
# ============================================================================

# À exécuter dans la septième cellule (optionnelle)
print("\n💾 SECTION 7: SAUVEGARDE DES MODÈLES")
print("=" * 50)

# Sauvegarde des modèles finaux
print("💾 Sauvegarde en cours...")

try:
    # Modèle initial
    if 'model_initial' in locals():
        model_initial.save_pretrained("./model_initial_final")
        print("✅ Modèle initial sauvegardé: ./model_initial_final")
    else:
        trainer.model.save_pretrained("./model_initial_final")
        print("✅ Modèle de base sauvegardé: ./model_initial_final")
    
    # Modèle réentraîné
    trainer_retrained.model.save_pretrained("./model_retrained_final")
    print("✅ Modèle réentraîné sauvegardé: ./model_retrained_final")
    
    # Processeur
    processor.save_pretrained("./processor_final")
    print("✅ Processeur sauvegardé: ./processor_final")
    
    print("\n🏁 WORKFLOW TERMINÉ!")
    print("📁 Fichiers générés:")
    print("   - ./model_initial_final/")
    print("   - ./model_retrained_final/")
    print("   - ./processor_final/")
    print("   - final_evaluation_report.csv")
    
    # Vérifier quels fichiers d'analyse existent
    import os
    analysis_files = [
        "initial_model_errors_characters.csv",
        "initial_model_errors_confusions.csv", 
        "test_evaluation_results.csv",
        "test_evaluation_errors.csv"
    ]
    
    existing_files = [f for f in analysis_files if os.path.exists(f)]
    if existing_files:
        print("   - " + "\n   - ".join(existing_files))
    
    print("\n🎯 RÉSULTATS FINAUX:")
    print(f"   WER: {results_initial['wer']:.4f} → {results_retrained['wer']:.4f} ({wer_improvement:+.4f})")
    print(f"   CER: {results_initial['cer']:.4f} → {results_retrained['cer']:.4f} ({cer_improvement:+.4f})")
    print(f"   {verdict}")
    
except Exception as e:
    print(f"❌ Erreur lors de la sauvegarde: {e}")
    print("⚠️ Certains modèles n'ont pas pu être sauvegardés")

print("\n🎊 ANALYSE TERMINÉE! Consultez le rapport final pour les détails complets.")